# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the metadata object and print its key properties
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Authors: {[a['@id'] for a in getattr(metadata, 'author', [])]}")
print(f"Distribution IDs: {[d['@id'] for d in getattr(metadata, 'distribution', [])]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section will display the datasets contained in the Croissant package. Each record set, field, and column is referenced by its `@id` as per FAIR principles.

In [ ]:
# List the available record sets in the dataset
record_sets = []
# Some Croissant datasets may have recordSet references under metadata.recordSet
if hasattr(metadata, 'recordSet') and len(getattr(metadata, 'recordSet', [])) > 0:
    record_sets = [rs['@id'] for rs in metadata.recordSet]
else:
    # If recordSet is empty, infer from other metadata or try common table names (from description)
    # Based on dataset description, we expect: 'table_1', 'table_2', 'table_3'
    record_sets = [
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table-1',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table-2',
        'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table-3'
    ]
print("Available records sets `@id`s:")
for rs in record_sets:
    print(f"- {rs}")

# Display a preview of records for each record set using mlcroissant
for rs_id in record_sets:
    print(f"\nRecords for Record Set @id: {rs_id}")
    try:
        for x in dataset.records(record_set=rs_id):
            print(x)
            break  # Preview only the first row
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

## 3. Data Extraction
Load data from each specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

The following code loads all record sets described above.

In [ ]:
# Extract data from each record set into a DataFrame using @id
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {record_set}")
        print("Columns:", dataframes[record_set].columns.tolist())
        print(dataframes[record_set].head(3), "\n")
    except Exception as e:
        print(f"Could not load DataFrame for {record_set}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note**: In this step, all field references use their `@id` as column names.

In [ ]:
# Choose Table 2 for hormone analysis as described in the dataset
table2_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/table-2'
df = dataframes.get(table2_id)
if df is not None:
    # List column @ids for selection
    print("Columns (Field @ids):")
    print(df.columns.tolist())
    
    # Example numeric hormone field: guess using 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field-serum_17ohp' as typical
    numeric_field_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field-serum_17ohp'  # 17-hydroxyprogesterone serum value
    # If column not present, pick any numeric field present
    if numeric_field_id not in df.columns:
        numeric_candidates = [col for col in df.columns if '17ohp' in col or 'value' in col or df[col].dtype in ['float64', 'int64']]
        if numeric_candidates:
            numeric_field_id = numeric_candidates[0]
            print(f"Fallback numeric field: {numeric_field_id}")

    threshold = 10
    # Filter for records above threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Group by adrenal CT result field (example field @id)
    group_field_id = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/field-adrenal_ct'
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Table 2 DataFrame not found for record set {table2_id}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected hormone field for Table 2 and visualize the relationship to adrenal CT findings, referencing column `@id`s.

In [ ]:
# Visualization
if df is not None:
    try:
        # Histogram for hormone values
        plt.figure(figsize=(7, 4))
        df[numeric_field_id].hist(bins=10, color='skyblue')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel("Value")
        plt.ylabel("Frequency")
        plt.show()

        # Boxplot grouped by adrenal CT result
        if group_field_id in df.columns:
            plt.figure(figsize=(7, 4))
            df.boxplot(column=numeric_field_id, by=group_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.suptitle('')
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
    except Exception as e:
        print(f"Could not create visualization: {e}")

## 6. Conclusion
In this notebook, we explored the FAIR² dataset on genotype–phenotype heterogeneity in non-classical 21-hydroxylase deficiency infertility cases using the `mlcroissant` library.

- We loaded dataset metadata and records, referencing each entity by its `@id`.
- We reviewed available record sets and extracted tabular data.
- We applied common EDA steps such as filtering, normalization, and grouping using column `@id`s.
- Visualizations revealed hormone level distributions and possible relationships with adrenal imaging findings.

This approach ensures reproducibility and interoperability, as all processing referenced the unique Croissant identifiers for fields and tables. The dataset is best used for academic and clinical illustration, not for predictive ML model training.